In [1]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler


DATA_PATH = "/content/gdrive/MyDrive/REU/PROJECT/DATA/"
DATA_PATH = "/home/jovyan/work/ner_congressional_records/DATA/"


# from google.colab import drive
# drive.mount('/content/gdrive')

In [2]:
annotated_data = pd.read_csv(DATA_PATH+"annotated_data.csv")

<h1>CONVERT TO BINARY LABELS</h1>

In [4]:
def org_labels(st):
    if st != "Individual" and st != "Candidate":
        return "Organization"
    return "Individual"

In [5]:
annotated_data['transactor_type'] = annotated_data['transactor_type'].apply(org_labels)
annotated_data = annotated_data[['full_name', 'transactor_type']].copy()

In [6]:
annotated_data.dropna(subset='full_name', inplace=True)

In [7]:
annotated_data['full_name'].sort_values()

9580                           citizens for boyle
8800                              mauck, shawn c 
360485                                !jayne 2012
95841                     ""rip"" stephen  wilson
323972    "a company, inc. phx portable restooms"
                           ...                   
353475                           zygmunt  roguski
94328                            zylphia  cummins
193258                                    zymages
433906                                zyra  brown
442010                   \tindependentvoting.org
Name: full_name, Length: 518286, dtype: object

<h1>Train, Test, split</h1>

In [8]:

def convert_bool(input_label):
    '''Converts to numerical encoding'''
    if input_label == 'Candidate' or input_label == 'Individual':
        return 1
    else:
        return 0

# encode transactor_type as numeric labels
annotated_data['label'] = np.vectorize(convert_bool)(annotated_data['transactor_type'])

# features and target for under-sampling
X = annotated_data[['full_name']]
y = annotated_data['label']

# apply random under-sampling to balance the classes
rus = RandomUnderSampler(random_state = 42)
X_resampled, y_resampled = rus.fit_resample(X, y)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.25, random_state=24)


train_set = X_train.copy().reset_index()

In [17]:
train_set['label']=y_train

In [19]:
train_set = X_train.copy()
train_set['label'] = y_train
train_set

,full_name,label
33997,james lanigan,1
363131,banners on the cheap.com,0
289259,committee to elect barry gillaspie,0
206818,otis albert,0
6077,mt. lebanon democratic committee c/o geoffrey ...,0
...,...,...
443751,melinda t bishop-morfin,1
18613,sharon e huie-lew,1
7657,"newtown democrats, regina gairo, treasurer",0
458214,driss ferza,1


In [20]:
test_set = X_test.copy()
test_set['label'] = y_test
test_set

,full_name,label
507706,quail creek crossing,0
225326,lyn t ward,1
141944,the governors,0
404422,briteverify,0
258567,u.s. american,0
...,...,...
228830,thomas russo,1
336128,colorado democratic party,0
300745,andreini-brophy anna,1
425221,jill blair,1


In [21]:
train_set.to_csv("train.csv")
test_set.to_csv("test.csv")

In [27]:
test = pd.read_csv(DATA_PATH+"test.csv")
test

,Unnamed: 0,full_name,label
0,507706,quail creek crossing,0
1,225326,lyn t ward,1
2,141944,the governors,0
3,404422,briteverify,0
4,258567,u.s. american,0
...,...,...,...
30647,109541,national pen company,0
30648,146726,colleen r floyd,1
30649,475604,robin cerillo,1
30650,482662,kenton rael,1
